# Clase 3 · 01 · Auto Loader → Bronze
Leemos una fuente abierta a futuras llegadas. El trigger procesa la parte disponible y termina.
El checkpoint conserva qué archivos ya ingresaron para la próxima ejecución.

In [ ]:
dbutils.widgets.text("student_id", "alumno01", "01 - Identificador")
dbutils.widgets.dropdown("scale", "small", ["test", "small", "demo"], "02 - Escala de clases 1 y 2")
dbutils.widgets.dropdown("expected_batch_id", "stream_001",
                         ["stream_001", "stream_002", "stream_003", "stream_004", "stream_005"], "03 - Etapa a validar")
dbutils.widgets.text("job_run_id", "interactive", "04 - ID de ejecución")

In [ ]:
%run ../common/config

In [ ]:
%run ../common/generate_stream_data

In [ ]:
%run ../common/streaming_support

In [ ]:
from pyspark.sql import functions as F
config = CourseConfig(spark.sql("SELECT current_catalog()").first()[0],
                      dbutils.widgets.get("student_id"), dbutils.widgets.get("scale"))
repetitions = STREAM_REPETITIONS[config.scale]
expected_batch_id = normalize_stream_batch(dbutils.widgets.get("expected_batch_id"))
job_run_id = dbutils.widgets.get("job_run_id").strip()[:100] or "interactive"
spark.conf.set("spark.sql.session.timeZone", "UTC")
require_stream_lab(spark, config)

## 1. Declarar la fuente streaming
Ingestamos JSON Lines como **TEXT**, conservando el texto original; el contrato JSON se aplica en Silver.
Auto Loader usa listado de archivos, sin notificaciones externas. Cada archivo tiene un nombre nuevo.
El `_metadata.file_path` permite rastrear el origen sin usar `input_file_name()`.

In [ ]:
files = (spark.readStream.format("cloudFiles")
         .option("cloudFiles.format", "text")
         .option("cloudFiles.useNotifications", "false")
         .schema("value STRING")
         .load(stream_input_directory(config)))
bronze = files.select(F.col("value").alias("raw_json"),
                      F.col("_metadata.file_path").alias("source_file"),
                      F.current_timestamp().alias("ingested_at"))
assert bronze.isStreaming
bronze.printSchema()

## 2. Escribir incrementalmente y esperar finalización
No se hace display sobre el DataFrame streaming: evitamos iniciar otra consulta y otro checkpoint.
La consulta escribe en Delta en append. La espera termina automáticamente o falla tras 10 minutos.

In [ ]:
query = (bronze.writeStream.format("delta").outputMode("append")
         .option("checkpointLocation", stream_checkpoint(config, "bronze"))
         .queryName(f"{config.schema}_bronze")
         .trigger(availableNow=True)
         .toTable(qualified_table(config, "bronze_stream_events")))
wait_and_record_stream(spark, config, query, "bronze", job_run_id)

In [ ]:
%sql
SELECT get_json_object(raw_json, '$.source_batch_id') AS source_batch_id,
       COUNT(*) AS physical_rows, COUNT(DISTINCT source_file) AS files
FROM bronze_stream_events GROUP BY get_json_object(raw_json, '$.source_batch_id')
ORDER BY source_batch_id

### Para observar
Reiniciar la consulta con el mismo checkpoint y sin archivos nuevos no debe agregar filas.
Un evento repetido dentro de otro archivo sí ingresa: la identidad del archivo y la del evento son diferentes.